In [12]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
import numpy as np


DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_best_customers.pickle")
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
#df = df[df["product_id"].isin(product_ids)]
df.drop(columns=["periodo"], inplace=True, errors="ignore")

In [13]:
numeric_df = df.select_dtypes(include=[np.number])
total_infs = np.isinf(numeric_df.values).sum()
print(f"Total infs: {total_infs}")
df[numeric_df.columns] = df[numeric_df.columns].replace([np.inf, -np.inf], np.nan)

Total infs: 10849


In [3]:
df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)
#df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M'))  # último día del mes
unique_fechas = df["fecha"].unique()
# aplico el to_timestamp('M') a las fechas unicas y despues reemplazo en el df
unique_fechas_transform = [x.to_timestamp('M') for x in unique_fechas]
df['fecha'] = df['fecha'].replace(unique_fechas, unique_fechas_transform)
del unique_fechas, unique_fechas_transform

/tmp/ipykernel_534352/105179847.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)
/tmp/ipykernel_534352/105179847.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['fecha'] = df['fecha'].replace(unique_fechas, unique_fechas_transform)


In [14]:
TEST_DATE = 33

def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index
df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)

test_index, train_index, train_scaler_index = get_indexes(df)
train_df = df.loc[train_index].copy().dropna(subset=['target'])
test_df = df.loc[test_index].copy()

/tmp/ipykernel_534352/1917849065.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)


# Modelo de serie de tiempo

In [21]:
static_features_df = pd.DataFrame({
    'cat1': train_df.groupby('serie_id')['cat1'].first(),
    'cat2': train_df.groupby('serie_id')['cat2'].first(),
    'cat3': train_df.groupby('serie_id')['cat3'].first(),
    'brand': train_df.groupby('serie_id')['brand'].first(),
    'sku_size': train_df.groupby('serie_id')['sku_size'].first(),
    "product_id": train_df.groupby('serie_id')['product_id'].first(),
    "customer_id": train_df.groupby('serie_id')['customer_id'].first(),
    "plan_precios_cuidados": train_df.groupby('serie_id')['plan_precios_cuidados'].first(),
}).reset_index()
# drop this from test)DF
train_df_customers = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size', "customer_id", "product_id", 'plan_precios_cuidados'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_customers.columns:
    if train_df_customers[column].dtype == "int16" or train_df_customers[column].dtype == "int8":
        train_df_customers[column] = train_df_customers[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_customers.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="serie_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()




predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    eval_metric="WAPE",  # Weighted Absolute Percentage Error
    horizon_weight=[0.5, 1]
)


predictor.fit(
    train_data,
    presets="medium_quality",
    #time_limit=600,
)
test_pred = predictor.predict(train_data)
test_pred

predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'serie_id'})
predictions = predictions.merge(test_df[['serie_id',"product_id", 'target']].drop_duplicates(), on='serie_id', how='left')
predictions = predictions.groupby('product_id').agg({
    'mean': 'sum',
    'target': 'sum',
}).reset_index()
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")
predictions

No path specified. Models will be saved in: "AutogluonModels/ag-20250628_170915"
Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250628_170915'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       6.21 GB / 15.32 GB (40.6%)
Disk Space Avail:   100.63 GB / 575.67 GB (17.5%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'freq': 'ME',
 'horizon_weight': array([[0.66666667, 1.33333333]]),
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_f

Total Absolute Error: 0.3347


,product_id,mean,target,abs_error
0,20001.0,909.811722,714.817200,194.994522
1,20002.0,793.721724,785.491150,8.230574
2,20003.0,418.951212,422.999878,4.048665
3,20004.0,204.808459,156.421143,48.387316
4,20005.0,257.720372,170.944656,86.775716
...,...,...,...,...
964,21263.0,0.000314,0.000000,0.000314
966,21265.0,-0.002164,0.045510,0.047674
967,21266.0,-0.002164,0.045510,0.047674
968,21267.0,0.002174,0.000000,0.002174


# Modelo tabular

In [6]:
TEST_DATE = 33

def get_indexes_tabular(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date-2]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index

test_index, train_index, train_scaler_index = get_indexes_tabular(df)



In [5]:
# pruebo transformando el target (ahora si es relevante en tabular)
import re
transformations = {
    "tn": [
        r"tn$",
        r"cust_request_qty_per_tn$",
        r"tn_lag_*",
        r"tn_rolling_mean_*",
        r"tn_rolling_max_*",
        r"tn_rolling_min_*",
        r"tn_.*_vendidas$",
        r"tn_agg*",
    ]
    + [r"stock_final$"]
    + [r"cust_request_tn_minus_tn$"]
    + [r"tn_diff_*"],
    "cust_request_qty": [
        r"cust_request_qty$",
        r"cust_request_qty_lag_*",
        r"cust_request_qty_rolling_mean_*",
        r"cust_request_qty_rolling_max_*",
        r"cust_request_qty_rolling_min_*",
        r"cust_request_qty_.*_vendidas$",
        r"cust_request_qty_agg*",
    ]
    + [r"cust_request_qty_diff_*"],
}

def scale_df(df, transformations, train_scaler_index):
    numeric_columns = df.select_dtypes(include=["float64", "float32", "int32", "int64"]).columns
    print(numeric_columns)
    df_scaled = df  # no hago copy intencionalmente
    train_scaler_df = df_scaled.loc[train_scaler_index]


    group_stats = train_scaler_df.groupby(["customer_id", "product_id"])[
        list(transformations.keys())
    ].agg(["mean", "std"]).reset_index()
    #print(prod_stats.head())
    group_stats.columns = [
        f"{col[0]}_{col[1]}" if col[1] else col[0] for col in group_stats.columns
    ]  # aplanar el multiindex de las columnas
    # replace infs with NaN
    group_stats = group_stats.replace([np.inf, -np.inf], np.nan)
    group_stats = group_stats.fillna(0)  # reemplazar NaN por
    print(group_stats.head())

    # Mergear las stats al df original
    df_scaled = df_scaled.merge(
        group_stats, on=["product_id", "customer_id"], how="left"
    )
    df_scaled = df_scaled.set_index(df.index)

    scaled_cols = {}
    for trainer, regex_cols in transformations.items():
        for col in regex_cols:
            # Usar regex para seleccionar las columnas que coinciden
            # chequear si la columna es un regex
            matching_cols = [c for c in numeric_columns if re.match(col, c)]
            if not matching_cols:
                continue  # Si no hay columnas que coincidan, saltar

            # Calcular la media y desviación estándar para cada
            print(f"Processing trainer: {trainer} with columns: {matching_cols}")
            # Escalar las columnas
            for col in matching_cols:
                scaled_cols[col + "_scaled"] = (df_scaled[col]) / df_scaled[
                    trainer + "_std"
                ]
                scaled_cols[col + "_scaled"].replace([np.inf, -np.inf], np.nan, inplace=True)
                scaled_cols[col + "_scaled"] = scaled_cols[col + "_scaled"].fillna(0)

    # Crear un DataFrame con todas las columnas escaladas
    scaled_df = pd.DataFrame(scaled_cols, index=df_scaled.index)

    # Concatenar de una sola vez
    df_scaled = pd.concat([df_scaled, scaled_df], axis=1)
    aux_cols = [col + "_mean" for col in list(transformations.keys())] + [
        col + "_std" for col in list(transformations.keys())
    ]
    df_scaled = df_scaled.drop(columns=aux_cols)
    return df_scaled, group_stats

df, group_stats = scale_df(df, transformations, train_scaler_index)
df

Index(['plan_precios_cuidados', 'cust_request_tn', 'tn', 'stock_final',
       'sku_size', 'coseno_fecha', 'seno_fecha', 'cust_request_tn_minus_tn',
       'tn_mult_cust_request_qty', 'tn_kama_indicator', 'tn_ppo', 'tn_roc',
       'tn_rsi', 'tn_stoch_rsi', 'tn_tsi', 'tn_ulcer_index', 'tn_dpo',
       'tn_macd_diff', 'tn_diff_1', 'tn_diff_2', 'tn_diff_3', 'tn_diff_5',
       'tn_diff_11', 'cust_request_qty_diff_1', 'cust_request_qty_diff_2',
       'cust_request_qty_diff_3', 'cust_request_qty_diff_5',
       'cust_request_qty_diff_11', 'tn_rolling_mean_6', 'tn_rolling_mean_12',
       'cust_request_qty_rolling_mean_6', 'cust_request_qty_rolling_mean_12',
       'tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_11', 'tn_lag_15',
       'cust_request_qty_lag_1', 'cust_request_qty_lag_2',
       'cust_request_qty_lag_3', 'cust_request_qty_lag_11',
       'cust_request_qty_lag_15', 'tn_rolling_mean_12_lag_1',
       'tn_rolling_mean_12_lag_2', 'tn_rolling_mean_12_lag_3',
       'tn_rolling_mean_

,product_id,fecha,customer_id,plan_precios_cuidados,cust_request_qty,cust_request_tn,tn,stock_final,cat1,cat2,...,cust_request_qty_rolling_min_6_scaled,cust_request_qty_rolling_min_12_scaled,cust_request_qty_cat3_vendidas_scaled,cust_request_qty_brand_vendidas_scaled,cust_request_qty_sku_size_vendidas_scaled,cust_request_qty_diff_1_scaled,cust_request_qty_diff_2_scaled,cust_request_qty_diff_3_scaled,cust_request_qty_diff_5_scaled,cust_request_qty_diff_11_scaled
0,20001,2017-01-31,10001,0.0,11,99.438606,99.438606,NaN,HC,ROPA LAVADO,...,0.0,0.0,251.154699,122.632413,186.717739,0.0,0.0,0.0,0.0,0.0
36,20001,2017-01-31,10002,0.0,17,38.683010,35.728062,NaN,HC,ROPA LAVADO,...,0.0,0.0,200.611068,97.953251,149.141725,0.0,0.0,0.0,0.0,0.0
72,20001,2017-01-31,10003,0.0,17,143.494263,143.494263,NaN,HC,ROPA LAVADO,...,0.0,0.0,335.029369,163.586269,249.073287,0.0,0.0,0.0,0.0,0.0
108,20001,2017-01-31,10004,0.0,9,184.729263,184.729263,NaN,HC,ROPA LAVADO,...,0.0,0.0,560.150206,273.507013,416.436485,0.0,0.0,0.0,0.0,0.0
144,20001,2017-01-31,10005,0.0,23,19.084070,19.084070,NaN,HC,ROPA LAVADO,...,0.0,0.0,211.772856,103.403267,157.439813,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316009,21276,2019-12-31,10006,0.0,0,0.000000,0.000000,1.05592,PC,PIEL1,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
316019,21276,2019-12-31,10007,0.0,0,0.000000,0.000000,1.05592,PC,PIEL1,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
316029,21276,2019-12-31,10008,0.0,0,0.000000,0.000000,1.05592,PC,PIEL1,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
316039,21276,2019-12-31,10009,0.0,0,0.000000,0.000000,1.05592,PC,PIEL1,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [6]:
df["target"] = df.groupby(['customer_id', 'product_id'])['tn_scaled'].shift(-2)

train_df_tabular = df.loc[train_index].copy().dropna(subset=['target'])
test_df_tabular = df.loc[test_index].copy()
del df, train_df, test_df

In [7]:
# modelo tabular baseline
from autogluon.tabular import TabularDataset, TabularPredictor

train_data_tabular = TabularDataset(train_df_tabular)
test_data_tabular = TabularDataset(test_df_tabular)

predictor_tabular = TabularPredictor(label="target").fit(
    train_data_tabular, 
    presets="medium_quality", 
    excluded_model_types=["RF", "XT"],
    time_limit=60
)
# exclude RF
y_pred_tabular = predictor_tabular.predict(test_data_tabular)
y_pred_tabular

No path specified. Models will be saved in: "AutogluonModels/ag-20250628_173921"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
Memory Avail:       6.52 GB / 15.32 GB (42.6%)
Disk Space Avail:   99.22 GB / 575.67 GB (17.2%)
Presets specified: ['medium_quality']
Beginning AutoGluon training ... Time limit = 60s
AutoGluon will save models to "/home/fede/programacion/labo3/AutogluonModels/ag-20250628_173921"
Train Data Rows:    291980
Train Data Columns: 173
Label Column:       target
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (74.83229064941406, 0.0, 0.9230700135231018, 1.102020025253296)
	If 'regression' is not the 

33        1.517210
69        1.571928
105       1.142076
141       1.271962
177       0.748167
            ...   
316007   -0.002510
316017   -0.003014
316027   -0.003014
316037   -0.003979
316047   -0.002616
Name: target, Length: 9720, dtype: float32

In [21]:
predictions_tabular = test_df_tabular[["product_id", "customer_id", "target"]].copy()
predictions_tabular["prediction"] = y_pred_tabular
predictions_tabular = predictions_tabular.merge(group_stats[["product_id", "customer_id", "tn_std"]], on=["product_id", "customer_id"], how="left")
predictions_tabular["prediction"] = predictions_tabular["prediction"] * predictions_tabular["tn_std"]
predictions_tabular["target"] = predictions_tabular["target"] * predictions_tabular["tn_std"]

predictions_tabular = predictions_tabular.groupby(['product_id']).agg({
    'target': 'sum',
    'prediction': 'sum'
}).reset_index()
predictions_tabular = predictions_tabular[predictions_tabular["product_id"].isin(product_ids)]
predictions_tabular["abs_error"] = abs(predictions_tabular['target'] - predictions_tabular['prediction'])
total_error_tabular = np.abs(predictions_tabular['target'] - predictions_tabular['prediction']).sum() / (predictions_tabular['target'].sum())
print(f"Total Absolute Error Tabular: {total_error_tabular:.4f}")
predictions_tabular


Total Absolute Error Tabular: 0.2627


,product_id,target,prediction,abs_error
0,20001,714.817200,702.369507,12.447693
1,20002,785.491150,559.537354,225.953796
2,20003,422.999878,305.841858,117.158020
3,20004,156.421143,151.282455,5.138687
4,20005,170.944656,169.588211,1.356445
...,...,...,...,...
964,21263,0.000000,0.000000,0.000000
966,21265,0.000000,0.000000,0.000000
967,21266,0.000000,0.000000,0.000000
968,21267,0.000000,0.000000,0.000000


In [ ]:
predictions = test_df_tabular[["product_id", "customer_id", "target"]].copy()
predictions["prediction"] = y_pred_tabular
predictions = predictions.groupby('product_id').agg({
    'prediction': 'sum',
    'target': 'sum',
}).reset_index()
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["abs_error"] = abs(predictions['target'] - predictions['prediction'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['prediction']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")
predictions


Total Absolute Error: 0.3853


,product_id,prediction,target,abs_error
0,20001,676.706360,714.817200,38.110840
1,20002,585.309875,785.491150,200.181274
2,20003,312.575806,422.999878,110.424072
3,20004,133.025879,156.421143,23.395264
4,20005,192.634949,170.944656,21.690292
...,...,...,...,...
964,21263,2.064403,0.000000,2.064403
966,21265,2.042377,0.045510,1.996867
967,21266,2.042446,0.045510,1.996936
968,21267,1.988036,0.000000,1.988036
